# OCR Regex Lab (info-scrap)

Notebook para extrair texto OCR de imagem via endpoint do serviço `info-scrap` e ajustar regex de parsing.

Fluxo:
1. Configurar URL base e caminho da imagem
2. Testar endpoint OCR (`/extract` primeiro, fallback `/product`)
3. Obter texto bruto
4. Aplicar regex e validar resultados

## Pré-requisitos
- Serviço `info-scrap` no ar
- Exemplo local (na raiz do projeto): `docker compose up -d info-scrap`
- Porta publicada no host: `8001` (conforme `docker-compose.yaml`)

In [4]:
import json
import os
import re
import subprocess
from pathlib import Path

BASE_URL = os.getenv('OCR_BASE_URL', 'http://localhost:8001')
IMAGE_PATH = os.getenv('OCR_IMAGE_PATH', '/home/v/Desktop/V_M/services/site/fotos/affiliate_full_20260312_002702.png')
ENDPOINT_CANDIDATES = ['/extract', '/product']  # /extract = endpoint OCR de texto bruto

print('BASE_URL    =', BASE_URL)
print('IMAGE_PATH  =', IMAGE_PATH)
print('IMAGE_EXISTS=', Path(IMAGE_PATH).exists())

BASE_URL    = http://localhost:8001
IMAGE_PATH  = /home/v/Desktop/V_M/services/site/fotos/affiliate_full_20260312_002702.png
IMAGE_EXISTS= True


In [5]:
def call_ocr_with_curl(base_url: str, endpoint: str, image_path: str, timeout_seconds: int = 90):
    """
    Chama o endpoint OCR via curl e retorna (status, data, raw_stdout, raw_stderr).
    """
    url = f"{base_url.rstrip('/')}/{endpoint.lstrip('/')}"
    cmd = [
        'curl',
        '-sS',
        '-X', 'POST',
        url,
        '-F', f"file=@{image_path}",
    ]

    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_seconds)
    stdout = (proc.stdout or '').strip()
    stderr = (proc.stderr or '').strip()

    if proc.returncode != 0:
        return proc.returncode, None, stdout, stderr

    try:
        data = json.loads(stdout) if stdout else {}
    except json.JSONDecodeError:
        data = {'_raw_text_response': stdout}

    return 0, data, stdout, stderr

In [6]:
# Tenta /extract primeiro (texto OCR), depois /product se necessário
selected_endpoint = None
ocr_response = None

if not Path(IMAGE_PATH).exists():
    raise FileNotFoundError(f'Imagem não encontrada: {IMAGE_PATH}')

for endpoint in ENDPOINT_CANDIDATES:
    status, data, stdout, stderr = call_ocr_with_curl(BASE_URL, endpoint, IMAGE_PATH)
    print(f'\nEndpoint {endpoint} -> status={status}')

    if stderr:
        print('stderr:', stderr[:300])

    if status == 0 and isinstance(data, dict):
        selected_endpoint = endpoint
        ocr_response = data
        print('Resposta recebida com sucesso.')
        break

if selected_endpoint is None:
    raise RuntimeError('Nenhum endpoint OCR respondeu com sucesso. Verifique container/porta e imagem.')

print('\nEndpoint selecionado:', selected_endpoint)
print('Chaves da resposta:', list(ocr_response.keys())[:20])
print('Preview JSON:')
print(json.dumps(ocr_response, ensure_ascii=False, indent=2)[:2000])


Endpoint /extract -> status=0
Resposta recebida com sucesso.

Endpoint selecionado: /extract
Chaves da resposta: ['success', 'text', 'lines', 'line_count']
Preview JSON:
{
  "success": true,
  "text": "Buscar produtos, ma Somente em Full Super\n\nConfira o envio para o seu endereço\n\nInclua seu CEP para verificar os custos e\nprazos de entrega precisos na busca\n\n[Dede = Mais tarde\nPáscoa\n\nVocê também pode estar interessado: omo em promocao - sabao liquido - omo 7 litros - sabao em po omo - sabao omo 7 litros - omo white\n\ntr\n\nMais opções que podem te interessar\n\nLava Roupas Líquido Profissional Sabão Em Pó Omo Lavagem Perfeita Lava Roupas Omo Sabão Liquido\nPerfect White 71 Omo Be Akg Lavagem Perfeita 7 Litros\n\nsss so n$sm46\nR$725 48% OFF R$ 4520 6% OFF\n\nBe s08\nR$ 8087 25% OFF no Pix\nou R$ 87,90 em outros meios\n\n10% OFF Linha de crédito\n\nCasa, Móveis e Decoração > Cuidado da Casa e Lavanderia > Produtos de Lavanderia > Sabão para Roupas > Sabao Liquido Vender um 

In [7]:
def extract_raw_ocr_text(payload: dict) -> str:
    """
    Extrai texto bruto da resposta OCR em formatos comuns.
    Prioriza campo 'text' (endpoint /extract).
    """
    if not isinstance(payload, dict):
        return ''

    if isinstance(payload.get('text'), str):
        return payload['text']

    if isinstance(payload.get('lines'), list):
        lines = [str(line).strip() for line in payload['lines'] if str(line).strip()]
        if lines:
            return '\n'.join(lines)

    # fallback para /product quando não vier texto completo
    pieces = []
    for key in ('title', 'price', 'oldPrice', 'stock'):
        val = payload.get(key)
        if isinstance(val, str) and val.strip():
            pieces.append(val.strip())

    return '\n'.join(pieces)

RAW_TEXT = extract_raw_ocr_text(ocr_response)
print('Tamanho do RAW_TEXT:', len(RAW_TEXT))
print('--- RAW_TEXT (preview 1200 chars) ---')
print(RAW_TEXT[:1200])

Tamanho do RAW_TEXT: 1213
--- RAW_TEXT (preview 1200 chars) ---
Buscar produtos, ma Somente em Full Super

Confira o envio para o seu endereço

Inclua seu CEP para verificar os custos e
prazos de entrega precisos na busca

[Dede = Mais tarde
Páscoa

Você também pode estar interessado: omo em promocao - sabao liquido - omo 7 litros - sabao em po omo - sabao omo 7 litros - omo white

tr

Mais opções que podem te interessar

Lava Roupas Líquido Profissional Sabão Em Pó Omo Lavagem Perfeita Lava Roupas Omo Sabão Liquido
Perfect White 71 Omo Be Akg Lavagem Perfeita 7 Litros

sss so n$sm46
R$725 48% OFF R$ 4520 6% OFF

Be s08
R$ 8087 25% OFF no Pix
ou R$ 87,90 em outros meios

10% OFF Linha de crédito

Casa, Móveis e Decoração > Cuidado da Casa e Lavanderia > Produtos de Lavanderia > Sabão para Roupas > Sabao Liquido Vender um igual | Compartilhar

| DISSENT) 2° em sabão para Roupas produto está disponível na sua

a

Lava Roupas Sabão Líquido OMO ego

Lavagem Perfeita 7 L
49 AM MAH (26281)



## Laboratório de Regex
Ajuste os padrões abaixo para seu caso e rode a célula.

In [8]:
REGEX_PATTERNS = {
    'precos_brl': r'(?:R\$|RS|R8|S\$)?\s*\d{1,3}(?:[.\s]\d{3})*(?:,\d{2})',
    'percentual_desc': r'\b\d{1,2}%\b',
    'parcelamento': r'\bem\s+\d{1,2}x\b',
}

def normalize_ocr_text(text: str) -> str:
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

clean_text = normalize_ocr_text(RAW_TEXT)

print('Texto normalizado (preview):')
print(clean_text[:600])

for name, pattern in REGEX_PATTERNS.items():
    matches = re.findall(pattern, clean_text, flags=re.IGNORECASE)
    print(f'\n[{name}] total={len(matches)}')
    print(matches[:20])

Texto normalizado (preview):
Buscar produtos, ma Somente em Full Super Confira o envio para o seu endereço Inclua seu CEP para verificar os custos e prazos de entrega precisos na busca [Dede = Mais tarde Páscoa Você também pode estar interessado: omo em promocao - sabao liquido - omo 7 litros - sabao em po omo - sabao omo 7 litros - omo white tr Mais opções que podem te interessar Lava Roupas Líquido Profissional Sabão Em Pó Omo Lavagem Perfeita Lava Roupas Omo Sabão Liquido Perfect White 71 Omo Be Akg Lavagem Perfeita 7 Litros sss so n$sm46 R$725 48% OFF R$ 4520 6% OFF Be s08 R$ 8087 25% OFF no Pix ou R$ 87,90 em outros 

[precos_brl] total=1
['R$ 87,90']

[percentual_desc] total=0
[]

[parcelamento] total=0
[]


In [9]:
# Comando curl equivalente (para rodar manualmente no terminal, se quiser):
manual_endpoint = selected_endpoint or '/extract'
curl_cmd = f"curl -sS -X POST '{BASE_URL}{manual_endpoint}' -F 'file=@{IMAGE_PATH}'"
print(curl_cmd)

curl -sS -X POST 'http://localhost:8001/extract' -F 'file=@/home/v/Desktop/V_M/services/site/fotos/affiliate_full_20260312_002702.png'


## Dica de ajuste fino
- Primeiro estabilize `RAW_TEXT` (imagem + endpoint)
- Depois refine `normalize_ocr_text`
- Por fim, itere os regex em `REGEX_PATTERNS`